## RWF-2000 + UCF-Crime Fighting + Bus Violence
### TEST ONLY — tanpa training

Notebook ini mengevaluasi sensitivitas **BiLSTM-Final terhadap temporal order**
pada tiga held-out dataset:

1. RWF-2000
2. UCF-Crime Fighting
3. Bus Violence

Tidak ada pooling antar-held-out dataset.

Protokol temporal-order mengikuti notebook referensi
`EffTempNet_BiLSTMFinal_TemporalOrder_TestOnly`:

```text
Original
Reverse
Shuffle 25%  → chunk_size = 4  pada T=16
Shuffle 50%  → chunk_size = 8  pada T=16
Shuffle 100% → chunk_size = 16 pada T=16
```

Setiap kondisi shuffle dijalankan **10 repeat** dengan permutation seed yang
deterministik dan independen dari training/checkpoint seed.

Jumlah dan isi frame tetap sama. Yang diubah hanya urutannya.

## Mengganti seed

Setelah path dataset benar, **cukup ubah satu variabel**:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [1]:
# Install dependency once per Colab session
!pip -q install timm


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import json
import random
import platform
import warnings
import zlib
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from timm import create_model
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm   :", timm.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))


Python : 3.13.15
PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

Untuk seed lain, **cukup ubah `RUN_SEED`** menjadi `0`, `42`, atau `123`.

Jangan mengubah:
- `SEQUENCE_LENGTH=16`
- `SHUFFLE_CHUNK_SIZES=[4,8,16]`
- `NUM_RANDOM_REPEATS=10`
- `PERMUTATION_BASE_SEED=20260809`
- preprocessing
- architecture
- held-out cache

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 0

VALID_SEEDS = (
    0,
    42,
    123,
)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}; "
        f"mendapat {RUN_SEED}."
    )

# ============================================================
# MODEL / CHECKPOINT
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/Models"
)

VARIANT = "bilstm_final"

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(
    seed: int,
) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    candidates = [
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_Seed{seed}_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_{seed}Seeds_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_{seed}Seed_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
    ]

    direct_matches = [
        path
        for path in candidates
        if path.exists()
    ]

    direct_matches = list(
        dict.fromkeys(
            direct_matches
        )
    )

    if len(
        direct_matches
    ) == 1:
        return direct_matches[
            0
        ]

    if len(
        direct_matches
    ) > 1:
        raise RuntimeError(
            "Ditemukan beberapa checkpoint BiLSTM-Final. "
            "Set CHECKPOINT_PATH_OVERRIDE secara eksplisit:\n"
            + "\n".join(
                str(
                    path
                )
                for path
                in direct_matches
            )
        )

    recursive_matches = sorted(
        MODEL_ROOT.glob(
            f"**/{VARIANT}/seed_{seed}/best.pt"
        )
    )

    recursive_matches = [
        path
        for path in recursive_matches
        if (
            "bilstmfinal"
            in str(
                path
            )
            .lower()
            .replace(
                "_",
                "",
            )
        )
    ]

    if len(
        recursive_matches
    ) == 1:
        return recursive_matches[
            0
        ]

    if len(
        recursive_matches
    ) == 0:
        raise FileNotFoundError(
            f"Checkpoint BiLSTM-Final seed {seed} tidak ditemukan "
            f"di {MODEL_ROOT}.\n"
            "Jika struktur folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint BiLSTM-Final yang ambigu. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(
            str(
                path
            )
            for path
            in recursive_matches
        )
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

# ============================================================
# EXACT MODEL / DATA PROTOCOL
# ============================================================
CLASS_NAMES = [
    "NonViolence",
    "Violence",
]

NUM_CLASSES = 2

SEQUENCE_LENGTH = 16

RAW_HEIGHT = 64
RAW_WIDTH = 64

BACKBONE_INPUT_SIZE = 224

LSTM_UNITS = 32

BIDIRECTIONAL = True

PREFER_LEGACY_CACHE = True

LEGACY_CACHE_COLOR_ORDER = "BGR"

AUTO_SCALE_LEGACY_CACHE = True

ALLOW_BUILD_CACHE = True

# ============================================================
# TEMPORAL-ORDER PROTOCOL — SAMA DENGAN NOTEBOOK REFERENSI
# ============================================================
SHUFFLE_CHUNK_SIZES = [
    4,
    8,
    16,
]

NUM_RANDOM_REPEATS = 10

# IMPORTANT:
# Ini dipertahankan dari notebook BiLSTM-Final TemporalOrder referensi.
PERMUTATION_BASE_SEED = 20260809

RUN_REVERSE = True

EVAL_BATCH_SIZE = 4

NUM_WORKERS = 2

USE_AMP = False

# ============================================================
# HELD-OUT DATASETS
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EXISTING_CACHE_DIR = (
    MODEL_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    MODEL_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

TEST_OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "BiLSTMFinal_TemporalOrder"
    / f"seed_{RUN_SEED}"
)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    SEQUENCE_LENGTH
    % max(
        SHUFFLE_CHUNK_SIZES
    )
    != 0
):
    raise ValueError(
        "Chunk size harus membagi SEQUENCE_LENGTH secara tepat."
    )

print(
    "RUN_SEED          :",
    RUN_SEED,
)

print(
    "Checkpoint        :",
    CHECKPOINT_PATH,
)

print(
    "Checkpoint exists :",
    CHECKPOINT_PATH.exists(),
)

print(
    "Output            :",
    TEST_OUTPUT_DIR,
)

print(
    "Random repeats    :",
    NUM_RANDOM_REPEATS,
)

print(
    "Chunk sizes       :",
    SHUFFLE_CHUNK_SIZES,
)

print(
    "Permutation seed  :",
    PERMUTATION_BASE_SEED,
)

RUN_SEED          : 0
Checkpoint        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_BiLSTMFinal_NoLS_Seed0_v1/bilstm_final/seed_0/best.pt
Checkpoint exists : True
Output            : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0
Random repeats    : 10
Chunk sizes       : [4, 8, 16]
Permutation seed  : 20260809


## 2. Reproducibility dan checkpoint loader

Checkpoint hanya dibaca. Notebook tidak membuat optimizer, scheduler, loss training, atau memanggil `.train()`.


In [5]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = bool(
    USE_AMP
    and DEVICE.type == "cuda"
)

AMP_DTYPE = (
    torch.bfloat16
    if (
        AMP_ENABLED
        and torch.cuda.is_bf16_supported()
    )
    else torch.float16
)


def amp_context():
    if not AMP_ENABLED:
        return nullcontext()

    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )


def set_eval_seed(
    seed: int,
) -> None:
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(
    worker_id: int,
):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def safe_torch_load(
    path: Path,
    map_location,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


set_eval_seed(
    RUN_SEED
)

print(
    "Device:",
    DEVICE,
    "| AMP:",
    AMP_ENABLED,
)

Device: cuda | AMP: False


## 3. Resolve / build held-out cache

Notebook menggunakan:
- RWF-2000;
- UCF-Crime Fighting;
- Bus Violence.

RWF memakai cache eksperimen lama bila tersedia. UCF dan Bus Violence memakai
shared held-out RGB-v2 cache yang sama dengan evaluator lain.

Cache builder dibuat **self-contained** agar tidak muncul:

```text
NameError: name 'SEQUENCE_LENGTH' is not defined
```

In [6]:
# ============================================================
# SELF-CONTAINED CACHE-BUILDER CONSTANTS
# ============================================================
# Python mengevaluasi default arguments saat function didefinisikan.
# Karena itu cache builder tidak bergantung pada SEQUENCE_LENGTH,
# RAW_HEIGHT, atau RAW_WIDTH dari cell lain.
_CACHE_BUILDER_SEQUENCE_LENGTH = 16
_CACHE_BUILDER_RAW_HEIGHT = 64
_CACHE_BUILDER_RAW_WIDTH = 64

if (
    "SEQUENCE_LENGTH" in globals()
    and int(
        globals()[
            "SEQUENCE_LENGTH"
        ]
    )
    != _CACHE_BUILDER_SEQUENCE_LENGTH
):
    raise RuntimeError(
        "Protocol mismatch: SEQUENCE_LENGTH harus 16."
    )

if (
    "RAW_HEIGHT" in globals()
    and int(
        globals()[
            "RAW_HEIGHT"
        ]
    )
    != _CACHE_BUILDER_RAW_HEIGHT
):
    raise RuntimeError(
        "Protocol mismatch: RAW_HEIGHT harus 64."
    )

if (
    "RAW_WIDTH" in globals()
    and int(
        globals()[
            "RAW_WIDTH"
        ]
    )
    != _CACHE_BUILDER_RAW_WIDTH
):
    raise RuntimeError(
        "Protocol mismatch: RAW_WIDTH harus 64."
    )

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = _CACHE_BUILDER_SEQUENCE_LENGTH,
    height: int = _CACHE_BUILDER_RAW_HEIGHT,
    width: int = _CACHE_BUILDER_RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": _CACHE_BUILDER_SEQUENCE_LENGTH,
                "height": _CACHE_BUILDER_RAW_HEIGHT,
                "width": _CACHE_BUILDER_RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{_CACHE_BUILDER_SEQUENCE_LENGTH},{_CACHE_BUILDER_RAW_HEIGHT},{_CACHE_BUILDER_RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

# ============================================================
# RESOLVE / AUDIT RWF, UCF, BUS VIOLENCE
# ============================================================
TEST_STORES = {}

dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    features = np.load(
        store[
            "feature_path"
        ],
        mmap_mode="r",
    )

    labels = np.load(
        store[
            "label_path"
        ],
        mmap_mode="r",
    ).astype(
        np.int64
    )

    expected_shape = (
        len(
            labels
        ),
        SEQUENCE_LENGTH,
        RAW_HEIGHT,
        RAW_WIDTH,
        3,
    )

    if features.shape != expected_shape:
        raise ValueError(
            f"{dataset_name}: cache shape {features.shape}; "
            f"expected {expected_shape}."
        )

    unique_labels = set(
        np.unique(
            labels
        ).tolist()
    )

    if unique_labels != {
        0,
        1,
    }:
        raise ValueError(
            f"{dataset_name}: held-out dataset harus mempunyai "
            f"dua kelas {{0,1}}, found {sorted(unique_labels)}."
        )

    TEST_STORES[
        dataset_name
    ] = (
        store[
            "feature_path"
        ],
        store[
            "label_path"
        ],
        store[
            "color_order"
        ],
        store.get(
            "paths_path"
        ),
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(
                    labels
                )
            ),
            "n_nonviolence": int(
                counts[
                    0
                ]
            ),
            "n_violence": int(
                counts[
                    1
                ]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
        }
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_df.to_csv(
    TEST_OUTPUT_DIR
    / "heldout_dataset_audit.csv",
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)

DATASET: UCF
✅ UCF: RGB-v2 cache

DATASET: BV
✅ BV: RGB-v2 cache


,dataset,n_total,n_nonviolence,n_violence,cache_source,color_order,feature_path,label_path
0,RWF,1998,1000,998,legacy,BGR,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
1,UCF,100,50,50,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
2,BV,1400,700,700,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...


## 4. Temporal permutation protocol

Untuk BiLSTM-Final dengan T=16:

### Shuffle 25%
```text
chunk_size = 4
[0..3] [4..7] [8..11] [12..15]
```

### Shuffle 50%
```text
chunk_size = 8
[0..7] [8..15]
```

### Shuffle 100%
```text
chunk_size = 16
[0..15]
```

Frame hanya diacak **di dalam chunk**. Tidak ada frame yang ditambah,
dihapus, atau diganti.

Reverse membalik sequence penuh secara deterministik.

In [7]:
IMAGENET_MEAN = (
    torch.tensor(
        [
            0.485,
            0.456,
            0.406,
        ],
        dtype=torch.float32,
    )
    .view(
        1,
        3,
        1,
        1,
    )
)

IMAGENET_STD = (
    torch.tensor(
        [
            0.229,
            0.224,
            0.225,
        ],
        dtype=torch.float32,
    )
    .view(
        1,
        3,
        1,
        1,
    )
)


def stable_dataset_code(
    dataset_name: str,
) -> int:
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def make_temporal_order(
    dataset_name: str,
    source_idx: int,
    mode: str,
    repeat_id: int = 0,
    chunk_size: Optional[int] = None,
) -> np.ndarray:
    base_order = np.arange(
        SEQUENCE_LENGTH,
        dtype=np.int64,
    )

    if mode == "original":
        return base_order

    if mode == "reverse":
        return base_order[
            ::-1
        ].copy()

    if mode != "shuffle":
        raise ValueError(
            f"Unknown mode: {mode}"
        )

    if chunk_size not in SHUFFLE_CHUNK_SIZES:
        raise ValueError(
            f"Invalid chunk_size={chunk_size}"
        )

    if (
        SEQUENCE_LENGTH
        % chunk_size
        != 0
    ):
        raise ValueError(
            f"T={SEQUENCE_LENGTH} tidak habis dibagi "
            f"chunk_size={chunk_size}"
        )

    # EXACT protocol from the source notebook:
    # deterministic per-video chunk permutation,
    # independent of checkpoint/training seed.
    seed_sequence = np.random.SeedSequence(
        [
            int(
                PERMUTATION_BASE_SEED
            ),
            int(
                repeat_id
            ),
            int(
                stable_dataset_code(
                    dataset_name
                )
            ),
            int(
                source_idx
            ),
            int(
                chunk_size
            ),
        ]
    )

    rng = np.random.default_rng(
        seed_sequence
    )

    order = base_order.copy()

    for start in range(
        0,
        SEQUENCE_LENGTH,
        chunk_size,
    ):
        stop = (
            start
            + chunk_size
        )

        order[
            start:stop
        ] = rng.permutation(
            order[
                start:stop
            ]
        )

    if not np.array_equal(
        np.sort(
            order
        ),
        base_order,
    ):
        raise AssertionError(
            "Temporal order bukan permutation lengkap."
        )

    return order


class TemporalOrderHeldOutDataset(
    Dataset
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
        paths_path: Optional[Path],
        mode: str,
        repeat_id: int = 0,
        chunk_size: Optional[int] = None,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        self.paths_path = (
            Path(
                paths_path
            )
            if paths_path is not None
            else None
        )

        self.mode = str(
            mode
        )

        self.repeat_id = int(
            repeat_id
        )

        self.chunk_size = chunk_size

        if (
            len(
                self.features
            )
            != len(
                self.labels
            )
        ):
            raise ValueError(
                f"Feature/label mismatch {dataset_name}: "
                f"{len(self.features)} vs {len(self.labels)}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def __getitem__(
        self,
        source_idx: int,
    ):
        frames = np.asarray(
            self.features[
                source_idx
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                source_idx
            ]
        )

        expected = (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if frames.shape != expected:
            raise ValueError(
                f"{self.dataset_name} idx={source_idx}: "
                f"shape={frames.shape}, expected={expected}"
            )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"NaN/Inf pada {self.dataset_name} "
                f"source_idx={source_idx}"
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = (
                    frames
                    / 255.0
                )

            else:
                raise ValueError(
                    f"Rentang cache tidak wajar: "
                    f"[{frame_min},{frame_max}]"
                )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        order = make_temporal_order(
            dataset_name=self.dataset_name,
            source_idx=int(
                source_idx
            ),
            mode=self.mode,
            repeat_id=self.repeat_id,
            chunk_size=self.chunk_size,
        )

        frames = frames[
            order
        ]

        clip = (
            torch.from_numpy(
                np.ascontiguousarray(
                    frames
                )
            )
            .permute(
                0,
                3,
                1,
                2,
            )
        )

        clip = (
            clip
            - IMAGENET_MEAN
        ) / IMAGENET_STD

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
            torch.from_numpy(
                order.copy()
            ),
        )


def build_test_loaders(
    mode: str,
    repeat_id: int = 0,
    chunk_size: Optional[int] = None,
):
    loaders = {}

    for dataset_name, (
        feature_path,
        label_path,
        color_order,
        paths_path,
    ) in TEST_STORES.items():
        dataset = TemporalOrderHeldOutDataset(
            dataset_name=dataset_name,
            feature_path=feature_path,
            label_path=label_path,
            color_order=color_order,
            paths_path=paths_path,
            mode=mode,
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        generator = torch.Generator()

        generator.manual_seed(
            PERMUTATION_BASE_SEED
        )

        loaders[
            dataset_name
        ] = DataLoader(
            dataset,
            batch_size=EVAL_BATCH_SIZE,
            shuffle=False,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=(
                DEVICE.type
                == "cuda"
            ),
            worker_init_fn=seed_worker,
            generator=generator,
            persistent_workers=(
                NUM_WORKERS
                > 0
            ),
        )

    return loaders


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if len(
        paths
    ) != expected_n:
        return None

    return [
        str(
            path
        )
        for path
        in paths
    ]

### Audit permutation sebelum inference

Cell berikut memastikan `shuffle_4/8/16` memang melakukan perturbation yang dimaksud.


In [8]:
audit_rows = []

for mode, chunk_size in [
    (
        "original",
        None,
    ),
    (
        "reverse",
        None,
    ),
    (
        "shuffle",
        4,
    ),
    (
        "shuffle",
        8,
    ),
    (
        "shuffle",
        16,
    ),
]:
    repeat_ids = (
        [
            0
        ]
        if mode != "shuffle"
        else [
            0,
            1,
            9,
        ]
    )

    for repeat_id in repeat_ids:
        first = make_temporal_order(
            dataset_name="RWF",
            source_idx=123,
            mode=mode,
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        second = make_temporal_order(
            dataset_name="RWF",
            source_idx=123,
            mode=mode,
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        if not np.array_equal(
            first,
            second,
        ):
            raise RuntimeError(
                "Permutation generator tidak deterministic."
            )

        audit_rows.append(
            {
                "mode": mode,
                "chunk_size": chunk_size,
                "repeat": repeat_id,
                "order_0based": first.tolist(),
            }
        )

permutation_audit_df = pd.DataFrame(
    audit_rows
)

permutation_audit_df.to_json(
    TEST_OUTPUT_DIR
    / "temporal_order_permutation_audit_examples.json",
    orient="records",
    indent=2,
)

display(
    permutation_audit_df
)

print(
    "✅ Temporal permutation audit PASS"
)

,mode,chunk_size,repeat,order_0based
0,original,NaN,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,reverse,NaN,0,"[15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, ..."
2,shuffle,4.0,0,"[2, 1, 0, 3, 7, 4, 5, 6, 8, 11, 9, 10, 15, 13,..."
3,shuffle,4.0,1,"[3, 2, 1, 0, 4, 5, 7, 6, 9, 10, 8, 11, 15, 14,..."
4,shuffle,4.0,9,"[1, 2, 3, 0, 7, 4, 6, 5, 11, 10, 9, 8, 13, 15,..."
5,shuffle,8.0,0,"[0, 1, 3, 5, 4, 2, 7, 6, 12, 13, 14, 15, 10, 9..."
6,shuffle,8.0,1,"[3, 7, 6, 1, 4, 2, 5, 0, 13, 11, 14, 9, 15, 8,..."
7,shuffle,8.0,9,"[4, 7, 2, 1, 3, 5, 0, 6, 14, 9, 13, 10, 11, 8,..."
8,shuffle,16.0,0,"[12, 1, 4, 7, 8, 10, 2, 6, 9, 0, 15, 14, 3, 11..."
9,shuffle,16.0,1,"[7, 15, 14, 1, 4, 11, 9, 2, 6, 12, 5, 8, 0, 10..."


✅ Temporal permutation audit PASS


## 5. Model BiLSTM-Final — arsitektur sama, tanpa training

```text
16 frames
→ EfficientFormerV2-S0 shared per frame
→ feature dropout 0.2
→ 1-layer BiLSTM hidden_size=32
→ concat final forward/backward states
→ classifier 64→256→128→64→32→2
```

Checkpoint dimuat menggunakan `strict=True`.

In [9]:
class EffTempNetBiLSTMFinal(
    nn.Module
):
    def __init__(
        self,
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        lstm_units: int = LSTM_UNITS,
    ):
        super().__init__()

        self.backbone = create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
            in_chans=3,
        )

        self.feature_dim = int(
            self.backbone.num_features
        )

        self.feature_dropout = nn.Dropout(
            0.2
        )

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=lstm_units,
            batch_first=True,
            bidirectional=True,
        )

        self.temporal_dim = (
            2
            * lstm_units
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                self.temporal_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.25
            ),

            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.25
            ),

            nn.Linear(
                128,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.2
            ),

            nn.Linear(
                64,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.2
            ),

            nn.Linear(
                32,
                num_classes,
            ),
        )

        self.attention = None

    def forward(
        self,
        clip: torch.Tensor,
    ):
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if seq_len != SEQUENCE_LENGTH:
            raise ValueError(
                f"Expected T={SEQUENCE_LENGTH}, got {seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected C=3, got {channels}"
            )

        x = clip.reshape(
            batch_size
            * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(
            x
        )

        if features.ndim == 4:
            features = (
                F.adaptive_avg_pool2d(
                    features,
                    output_size=1,
                )
                .flatten(
                    1
                )
            )

        elif features.ndim != 2:
            features = features.flatten(
                1
            )

        features = features.view(
            batch_size,
            seq_len,
            -1,
        )

        features = self.feature_dropout(
            features
        )

        (
            _,
            (
                h_n,
                _,
            ),
        ) = self.lstm(
            features
        )

        context = torch.cat(
            [
                h_n[
                    -2
                ],
                h_n[
                    -1
                ],
            ],
            dim=1,
        )

        logits = self.classifier(
            context
        )

        return logits


if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak mempunyai model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if "variant" in checkpoint_config:
    if str(
        checkpoint_config[
            "variant"
        ]
    ) != VARIANT:
        raise RuntimeError(
            "Checkpoint variant mismatch."
        )

if "seed" in checkpoint_config:
    if int(
        checkpoint_config[
            "seed"
        ]
    ) != RUN_SEED:
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )

if "sequence_length" in checkpoint_config:
    if int(
        checkpoint_config[
            "sequence_length"
        ]
    ) != SEQUENCE_LENGTH:
        raise RuntimeError(
            "Checkpoint sequence_length mismatch."
        )

model = EffTempNetBiLSTMFinal().to(
    DEVICE
)

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

if (
    model.lstm.hidden_size
    != LSTM_UNITS
):
    raise RuntimeError(
        "BiLSTM hidden size mismatch."
    )

if not model.lstm.bidirectional:
    raise RuntimeError(
        "BiLSTM harus bidirectional."
    )

if model.attention is not None:
    raise RuntimeError(
        "BiLSTM-Final tidak menggunakan temporal attention."
    )

with torch.inference_mode():
    dummy = torch.zeros(
        2,
        SEQUENCE_LENGTH,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if tuple(
    dummy_logits.shape
) != (
    2,
    NUM_CLASSES,
):
    raise RuntimeError(
        f"Unexpected output shape: "
        f"{tuple(dummy_logits.shape)}"
    )

print(
    "✅ Strict BiLSTM-Final checkpoint/model audit PASS"
)

print(
    "Epoch checkpoint:",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)

print(
    "Best score:",
    checkpoint.get(
        "best_score",
        "unknown",
    ),
)

print(
    "Load result:",
    load_result,
)

print(
    "Trainable params:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
)

del (
    dummy,
    dummy_logits,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Strict BiLSTM-Final checkpoint/model audit PASS
Epoch checkpoint: 14
Best score: 0.8812722919822154
Load result: <All keys matched successfully>
Trainable params: 0


## 6. Metrics

Metrik disimpan lengkap:
- Accuracy;
- Macro Precision;
- Macro Recall;
- Macro F1;
- ROC-AUC;
- Average Precision;
- Precision/Recall/F1 per kelas NonViolence dan Violence.


In [10]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    result = {
        "n": int(
            len(
                y_true
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "precision_nonviolence": float(
            precision_cls[
                0
            ]
        ),

        "recall_nonviolence": float(
            recall_cls[
                0
            ]
        ),

        "f1_nonviolence": float(
            f1_cls[
                0
            ]
        ),

        "precision_violence": float(
            precision_cls[
                1
            ]
        ),

        "recall_violence": float(
            recall_cls[
                1
            ]
        ),

        "f1_violence": float(
            f1_cls[
                1
            ]
        ),
    }

    if len(
        np.unique(
            y_true
        )
    ) == 2:
        result[
            "roc_auc"
        ] = float(
            roc_auc_score(
                y_true,
                p_violence,
            )
        )

        result[
            "average_precision"
        ] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )

    else:
        result[
            "roc_auc"
        ] = float(
            "nan"
        )

        result[
            "average_precision"
        ] = float(
            "nan"
        )

    return result


@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_orders = []

    for (
        clips,
        labels,
        indices,
        orders,
    ) in tqdm(
        loader,
        leave=False,
        desc="Test",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            logits = model(
                clips
            )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_orders.extend(
            orders.numpy().tolist()
        )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64,
    )

    probs = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    indices = np.asarray(
        all_indices,
        dtype=np.int64,
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probs[
            :,
            1
        ],
    )

    return (
        metrics,
        indices,
        y_true,
        y_pred,
        probs,
        all_orders,
    )

## 7. Evaluation runner

Setiap held-out dataset dievaluasi **terpisah**.

Tidak ada `ALL_TESTS_POOLED`.

Prediction CSV menyimpan `sample_key`, permutation, `y_true`, `y_pred`,
`is_correct`, dan class probabilities agar dapat dipakai untuk analisis paired.

In [11]:
def scenario_name(
    mode: str,
    chunk_size: Optional[int],
) -> str:
    if mode == "original":
        return "original"

    if mode == "reverse":
        return "reverse"

    return f"shuffle_{chunk_size}"


def evaluate_scenario(
    mode: str,
    repeat_id: int,
    chunk_size: Optional[int] = None,
):
    scenario = scenario_name(
        mode,
        chunk_size,
    )

    loaders = build_test_loaders(
        mode=mode,
        repeat_id=repeat_id,
        chunk_size=chunk_size,
    )

    metric_rows = []

    prediction_frames = []

    for (
        dataset_name,
        loader,
    ) in loaders.items():
        print(
            f"{scenario:12s} | "
            f"repeat={repeat_id:02d} | "
            f"{dataset_name}"
        )

        (
            metrics,
            indices,
            y_true,
            y_pred,
            probs,
            orders,
        ) = evaluate_loader(
            model,
            loader,
        )

        metric_rows.append(
            {
                "checkpoint_seed": RUN_SEED,
                "scenario": scenario,
                "mode": mode,
                "chunk_size": chunk_size,
                "repeat": repeat_id,
                "dataset": dataset_name,
                **metrics,
            }
        )

        (
            feature_path,
            label_path,
            color_order,
            paths_path,
        ) = TEST_STORES[
            dataset_name
        ]

        source_paths = load_paths_if_available(
            paths_path,
            expected_n=len(
                indices
            ),
        )

        if source_paths is None:
            sample_keys = [
                f"{dataset_name}::{int(index)}"
                for index in indices
            ]

            video_paths = [
                ""
                for _ in indices
            ]

        else:
            video_paths = [
                source_paths[
                    int(
                        index
                    )
                ]
                for index in indices
            ]

            sample_keys = [
                Path(
                    path
                ).as_posix()
                for path in video_paths
            ]

        condition_predictions_df = pd.DataFrame(
            {
                "checkpoint_seed": RUN_SEED,
                "scenario": scenario,
                "mode": mode,
                "chunk_size": chunk_size,
                "repeat": repeat_id,
                "dataset": dataset_name,
                "sample_index": indices,
                "sample_key": sample_keys,
                "video_path": video_paths,
                "permutation_0based": [
                    json.dumps(
                        order
                    )
                    for order in orders
                ],
                "y_true": y_true,
                "y_pred": y_pred,
                "is_correct": (
                    y_true
                    == y_pred
                ).astype(
                    np.int64
                ),
                "p_nonviolence": probs[
                    :,
                    0
                ],
                "p_violence": probs[
                    :,
                    1
                ],
            }
        )

        if condition_predictions_df[
            "sample_key"
        ].duplicated().any():
            raise RuntimeError(
                f"Duplicate sample_key: "
                f"{dataset_name}, {scenario}, repeat={repeat_id}."
            )

        prediction_frames.append(
            condition_predictions_df
        )

    metrics_df = pd.DataFrame(
        metric_rows
    )

    scenario_predictions_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    return (
        metrics_df,
        scenario_predictions_df,
    )

## 8. ORIGINAL

Original chronology dievaluasi satu kali dan menjadi reference untuk semua
delta:

\[
\Delta M = M_{\mathrm{Original}} - M_{\mathrm{Perturbed}}
\]

Nilai positif berarti metric turun setelah urutan temporal diubah.

In [12]:
all_metric_frames = []

all_prediction_frames = []

original_metrics, original_predictions = evaluate_scenario(
    mode="original",
    repeat_id=0,
    chunk_size=None,
)

all_metric_frames.append(
    original_metrics
)

all_prediction_frames.append(
    original_predictions
)

display(
    original_metrics[
        [
            "dataset",
            "average_precision",
            "roc_auc",
            "f1_macro",
            "accuracy",
            "balanced_accuracy",
        ]
    ].round(
        5
    )
)

original_metrics.to_csv(
    TEST_OUTPUT_DIR
    / f"metrics_original_seed{RUN_SEED}.csv",
    index=False,
)

original_predictions.to_csv(
    TEST_OUTPUT_DIR
    / f"predictions_original_seed{RUN_SEED}.csv",
    index=False,
)

print(
    "✅ Original held-out evaluation selesai."
)

original     | repeat=00 | RWF


Test:   0%|          | 0/500 [00:00<?, ?it/s]

original     | repeat=00 | UCF


Test:   0%|          | 0/25 [00:00<?, ?it/s]

original     | repeat=00 | BV


Test:   0%|          | 0/350 [00:00<?, ?it/s]

,dataset,average_precision,roc_auc,f1_macro,accuracy,balanced_accuracy
0,RWF,0.63001,0.64300,0.61192,0.61361,0.61355
1,UCF,0.69530,0.61640,0.60728,0.62000,0.62000
2,BV,0.49025,0.49285,0.48302,0.50071,0.50071


✅ Original held-out evaluation selesai.


## 9. Reverse-order diagnostic

Reverse membalik seluruh sequence:

```text
[0,1,...,15] → [15,14,...,0]
```

Semua frame tetap identik; hanya arahnya yang dibalik.

In [13]:
if RUN_REVERSE:
    (
        reverse_metrics,
        reverse_predictions,
    ) = evaluate_scenario(
        mode="reverse",
        repeat_id=0,
        chunk_size=None,
    )

    all_metric_frames.append(
        reverse_metrics
    )

    all_prediction_frames.append(
        reverse_predictions
    )

    reverse_metrics.to_csv(
        TEST_OUTPUT_DIR
        / f"metrics_reverse_seed{RUN_SEED}.csv",
        index=False,
    )

    reverse_predictions.to_csv(
        TEST_OUTPUT_DIR
        / f"predictions_reverse_seed{RUN_SEED}.csv",
        index=False,
    )

    display(
        reverse_metrics[
            [
                "dataset",
                "average_precision",
                "roc_auc",
                "f1_macro",
                "accuracy",
                "balanced_accuracy",
            ]
        ].round(
            5
        )
    )

reverse      | repeat=00 | RWF


Test:   0%|          | 0/500 [00:00<?, ?it/s]

reverse      | repeat=00 | UCF


Test:   0%|          | 0/25 [00:00<?, ?it/s]

reverse      | repeat=00 | BV


Test:   0%|          | 0/350 [00:00<?, ?it/s]

,dataset,average_precision,roc_auc,f1_macro,accuracy,balanced_accuracy
0,RWF,0.63354,0.64393,0.60360,0.60561,0.60554
1,UCF,0.71820,0.64880,0.58056,0.59000,0.59000
2,BV,0.50136,0.50387,0.48510,0.50286,0.50286


## 10. Chunk-wise temporal shuffling

Setiap kondisi:
- `shuffle_4`
- `shuffle_8`
- `shuffle_16`

dijalankan sebanyak **10 repeat**.

In [14]:
for chunk_size in SHUFFLE_CHUNK_SIZES:
    for repeat_id in range(
        NUM_RANDOM_REPEATS
    ):
        (
            condition_metrics_df,
            condition_predictions_df,
        ) = evaluate_scenario(
            mode="shuffle",
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        all_metric_frames.append(
            condition_metrics_df
        )

        all_prediction_frames.append(
            condition_predictions_df
        )

        scenario = (
            f"shuffle_{chunk_size}"
        )

        condition_metrics_df.to_csv(
            TEST_OUTPUT_DIR
            / (
                f"metrics_{scenario}_"
                f"repeat{repeat_id:02d}_seed{RUN_SEED}.csv"
            ),
            index=False,
        )

        condition_predictions_df.to_csv(
            TEST_OUTPUT_DIR
            / (
                f"predictions_{scenario}_"
                f"repeat{repeat_id:02d}_seed{RUN_SEED}.csv"
            ),
            index=False,
        )

print(
    "✅ Semua temporal-order tests selesai."
)

shuffle_4    | repeat=00 | RWF


Test:   0%|          | 0/500 [00:00<?, ?it/s]

shuffle_4    | repeat=00 | UCF


Test:   0%|          | 0/25 [00:00<?, ?it/s]

shuffle_4    | repeat=00 | BV


...

shuffle_16   | repeat=09 | UCF


Test:   0%|          | 0/25 [00:00<?, ?it/s]

shuffle_16   | repeat=09 | BV


Test:   0%|          | 0/350 [00:00<?, ?it/s]

✅ Semua temporal-order tests selesai.


## 11. Summary mean ± sampling std dan Δ terhadap Original

`std_sampling` pada shuffle menggambarkan variasi akibat random temporal
permutation, bukan variasi training seed.

AP adalah metric utama. Delta AP juga disimpan langsung dalam poin persentase.

In [15]:
all_metrics = pd.concat(
    all_metric_frames,
    ignore_index=True,
)

all_predictions = pd.concat(
    all_prediction_frames,
    ignore_index=True,
)

all_metrics_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_metrics_all_repeats_seed{RUN_SEED}.csv"
)

all_predictions_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_predictions_all_repeats_seed{RUN_SEED}.csv"
)

all_metrics.to_csv(
    all_metrics_path,
    index=False,
)

all_predictions.to_csv(
    all_predictions_path,
    index=False,
)

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "roc_auc",
    "average_precision",
    "precision_nonviolence",
    "recall_nonviolence",
    "f1_nonviolence",
    "precision_violence",
    "recall_violence",
    "f1_violence",
]

summary_rows = []

for (
    dataset,
    scenario,
), group in all_metrics.groupby(
    [
        "dataset",
        "scenario",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset,
        "scenario": scenario,
        "num_repeats": int(
            len(
                group
            )
        ),
    }

    for metric_name in metric_columns:
        values = group[
            metric_name
        ].astype(
            float
        )

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(
                values
            )
            > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "dataset",
        "scenario",
    ]
)

summary_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

# ============================================================
# DELTA RELATIVE TO ORIGINAL
# Δ > 0 = performance decrease after perturbation
# ============================================================
original_lookup = (
    all_metrics[
        all_metrics[
            "scenario"
        ]
        == "original"
    ]
    .set_index(
        "dataset"
    )
)

delta_rows = []

for _, row in all_metrics.iterrows():
    if row[
        "scenario"
    ] == "original":
        continue

    dataset = row[
        "dataset"
    ]

    if dataset not in original_lookup.index:
        continue

    original = original_lookup.loc[
        dataset
    ]

    delta_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset,
            "scenario": row[
                "scenario"
            ],
            "repeat": int(
                row[
                    "repeat"
                ]
            ),
            "delta_accuracy_orig_minus_perturbed": float(
                original[
                    "accuracy"
                ]
                - row[
                    "accuracy"
                ]
            ),
            "delta_balanced_accuracy_orig_minus_perturbed": float(
                original[
                    "balanced_accuracy"
                ]
                - row[
                    "balanced_accuracy"
                ]
            ),
            "delta_f1_macro_orig_minus_perturbed": float(
                original[
                    "f1_macro"
                ]
                - row[
                    "f1_macro"
                ]
            ),
            "delta_auc_orig_minus_perturbed": float(
                original[
                    "roc_auc"
                ]
                - row[
                    "roc_auc"
                ]
            ),
            "delta_ap_orig_minus_perturbed": float(
                original[
                    "average_precision"
                ]
                - row[
                    "average_precision"
                ]
            ),
            "delta_f1_pp": float(
                100.0
                * (
                    original[
                        "f1_macro"
                    ]
                    - row[
                        "f1_macro"
                    ]
                )
            ),
            "delta_auc_pp": float(
                100.0
                * (
                    original[
                        "roc_auc"
                    ]
                    - row[
                        "roc_auc"
                    ]
                )
            ),
            "delta_ap_pp": float(
                100.0
                * (
                    original[
                        "average_precision"
                    ]
                    - row[
                        "average_precision"
                    ]
                )
            ),
        }
    )

delta_df = pd.DataFrame(
    delta_rows
)

delta_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_delta_all_repeats_seed{RUN_SEED}.csv"
)

delta_df.to_csv(
    delta_path,
    index=False,
)

delta_summary_rows = []

for (
    dataset,
    scenario,
), group in delta_df.groupby(
    [
        "dataset",
        "scenario",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset,
        "scenario": scenario,
        "num_repeats": int(
            len(
                group
            )
        ),
    }

    for column in [
        "delta_accuracy_orig_minus_perturbed",
        "delta_balanced_accuracy_orig_minus_perturbed",
        "delta_f1_macro_orig_minus_perturbed",
        "delta_auc_orig_minus_perturbed",
        "delta_ap_orig_minus_perturbed",
        "delta_f1_pp",
        "delta_auc_pp",
        "delta_ap_pp",
    ]:
        values = group[
            column
        ].astype(
            float
        )

        row[
            f"{column}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{column}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(
                values
            )
            > 1
            else 0.0
        )

        row[
            f"{column}_min"
        ] = float(
            values.min()
        )

        row[
            f"{column}_max"
        ] = float(
            values.max()
        )

    delta_summary_rows.append(
        row
    )

delta_summary_df = pd.DataFrame(
    delta_summary_rows
).sort_values(
    [
        "dataset",
        "scenario",
    ]
)

delta_summary_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_delta_summary_seed{RUN_SEED}.csv"
)

delta_summary_df.to_csv(
    delta_summary_path,
    index=False,
)

print(
    "=== TEMPORAL-ORDER SUMMARY ==="
)

display(
    summary_df[
        [
            "dataset",
            "scenario",
            "num_repeats",
            "average_precision_mean",
            "average_precision_std_sampling",
            "roc_auc_mean",
            "roc_auc_std_sampling",
            "f1_macro_mean",
            "f1_macro_std_sampling",
        ]
    ].round(
        5
    )
)

print(
    "=== Δ VS ORIGINAL — POIN PERSENTASE ==="
)

display(
    delta_summary_df[
        [
            "dataset",
            "scenario",
            "num_repeats",
            "delta_ap_pp_mean",
            "delta_ap_pp_std_sampling",
            "delta_auc_pp_mean",
            "delta_f1_pp_mean",
        ]
    ].round(
        4
    )
)

print(
    "Saved metrics:",
    all_metrics_path,
)

print(
    "Saved predictions:",
    all_predictions_path,
)

print(
    "Saved summary:",
    summary_path,
)

print(
    "Saved deltas:",
    delta_path,
)

print(
    "Saved delta summary:",
    delta_summary_path,
)

=== TEMPORAL-ORDER SUMMARY ===


,dataset,scenario,num_repeats,average_precision_mean,average_precision_std_sampling,roc_auc_mean,roc_auc_std_sampling,f1_macro_mean,f1_macro_std_sampling
0,BV,original,1,0.49025,0.00000,0.49285,0.00000,0.48302,0.00000
1,BV,reverse,1,0.50136,0.00000,0.50387,0.00000,0.48510,0.00000
2,BV,shuffle_16,10,0.49652,0.00221,0.49542,0.00181,0.47876,0.00353
3,BV,shuffle_4,10,0.48847,0.00134,0.49109,0.00082,0.48384,0.00242
4,BV,shuffle_8,10,0.49118,0.00155,0.49092,0.00144,0.48187,0.00406
5,RWF,original,1,0.63001,0.00000,0.64300,0.00000,0.61192,0.00000
6,RWF,reverse,1,0.63354,0.00000,0.64393,0.00000,0.60360,0.00000
7,RWF,shuffle_16,10,0.63143,0.00284,0.64260,0.00218,0.60832,0.00293
8,RWF,shuffle_4,10,0.62911,0.00119,0.64169,0.00081,0.60713,0.00342
9,RWF,shuffle_8,10,0.63052,0.00231,0.64222,0.00171,0.60780,0.00338


=== Δ VS ORIGINAL — POIN PERSENTASE ===


,dataset,scenario,num_repeats,delta_ap_pp_mean,delta_ap_pp_std_sampling,delta_auc_pp_mean,delta_f1_pp_mean
0,BV,reverse,1,-1.1113,0.0000,-1.1014,-0.2078
1,BV,shuffle_16,10,-0.6270,0.2206,-0.2569,0.4258
2,BV,shuffle_4,10,0.1773,0.1339,0.1766,-0.0823
3,BV,shuffle_8,10,-0.0932,0.1552,0.1935,0.1146
4,RWF,reverse,1,-0.3525,0.0000,-0.0930,0.8316
5,RWF,shuffle_16,10,-0.1417,0.2843,0.0395,0.3601
6,RWF,shuffle_4,10,0.0906,0.1189,0.1307,0.4787
7,RWF,shuffle_8,10,-0.0509,0.2313,0.0780,0.4118
8,UCF,reverse,1,-2.2896,0.0000,-3.2400,2.6713
9,UCF,shuffle_16,10,-2.5174,0.9755,-4.1560,0.7220


Saved metrics: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_metrics_all_repeats_seed0.csv
Saved predictions: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_predictions_all_repeats_seed0.csv
Saved summary: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_summary_seed0.csv
Saved deltas: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_delta_all_repeats_seed0.csv
Saved delta summary: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_delta_summary_seed0.csv


## 12. Ringkasan per held-out dataset

RWF, UCF, dan Bus Violence ditampilkan terpisah.

Tidak ada pooling antar-domain.

In [16]:
scenario_order = [
    "original",
    "reverse",
    "shuffle_4",
    "shuffle_8",
    "shuffle_16",
]

for dataset_name in HELDOUT_SPECS.keys():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    dataset_summary = summary_df[
        summary_df[
            "dataset"
        ]
        == dataset_name
    ].copy()

    dataset_summary[
        "scenario"
    ] = pd.Categorical(
        dataset_summary[
            "scenario"
        ],
        categories=scenario_order,
        ordered=True,
    )

    dataset_summary = dataset_summary.sort_values(
        "scenario"
    )

    display(
        dataset_summary[
            [
                "scenario",
                "average_precision_mean",
                "average_precision_std_sampling",
                "roc_auc_mean",
                "f1_macro_mean",
            ]
        ].round(
            5
        )
    )

    dataset_delta = delta_summary_df[
        delta_summary_df[
            "dataset"
        ]
        == dataset_name
    ].copy()

    dataset_delta[
        "scenario"
    ] = pd.Categorical(
        dataset_delta[
            "scenario"
        ],
        categories=scenario_order[
            1:
        ],
        ordered=True,
    )

    dataset_delta = dataset_delta.sort_values(
        "scenario"
    )

    display(
        dataset_delta[
            [
                "scenario",
                "delta_ap_pp_mean",
                "delta_ap_pp_std_sampling",
                "delta_auc_pp_mean",
                "delta_f1_pp_mean",
            ]
        ].round(
            4
        )
    )


DATASET: RWF


,scenario,average_precision_mean,average_precision_std_sampling,roc_auc_mean,f1_macro_mean
5,original,0.63001,0.00000,0.64300,0.61192
6,reverse,0.63354,0.00000,0.64393,0.60360
8,shuffle_4,0.62911,0.00119,0.64169,0.60713
9,shuffle_8,0.63052,0.00231,0.64222,0.60780
7,shuffle_16,0.63143,0.00284,0.64260,0.60832


,scenario,delta_ap_pp_mean,delta_ap_pp_std_sampling,delta_auc_pp_mean,delta_f1_pp_mean
4,reverse,-0.3525,0.0000,-0.0930,0.8316
6,shuffle_4,0.0906,0.1189,0.1307,0.4787
7,shuffle_8,-0.0509,0.2313,0.0780,0.4118
5,shuffle_16,-0.1417,0.2843,0.0395,0.3601



DATASET: UCF


,scenario,average_precision_mean,average_precision_std_sampling,roc_auc_mean,f1_macro_mean
10,original,0.69530,0.00000,0.61640,0.60728
11,reverse,0.71820,0.00000,0.64880,0.58056
13,shuffle_4,0.70462,0.00867,0.63008,0.58613
14,shuffle_8,0.70569,0.00371,0.64212,0.59152
12,shuffle_16,0.72048,0.00976,0.65796,0.60006


,scenario,delta_ap_pp_mean,delta_ap_pp_std_sampling,delta_auc_pp_mean,delta_f1_pp_mean
8,reverse,-2.2896,0.0000,-3.240,2.6713
10,shuffle_4,-0.9315,0.8674,-1.368,2.1142
11,shuffle_8,-1.0386,0.3713,-2.572,1.5751
9,shuffle_16,-2.5174,0.9755,-4.156,0.7220



DATASET: BV


,scenario,average_precision_mean,average_precision_std_sampling,roc_auc_mean,f1_macro_mean
0,original,0.49025,0.00000,0.49285,0.48302
1,reverse,0.50136,0.00000,0.50387,0.48510
3,shuffle_4,0.48847,0.00134,0.49109,0.48384
4,shuffle_8,0.49118,0.00155,0.49092,0.48187
2,shuffle_16,0.49652,0.00221,0.49542,0.47876


,scenario,delta_ap_pp_mean,delta_ap_pp_std_sampling,delta_auc_pp_mean,delta_f1_pp_mean
0,reverse,-1.1113,0.0000,-1.1014,-0.2078
2,shuffle_4,0.1773,0.1339,0.1766,-0.0823
3,shuffle_8,-0.0932,0.1552,0.1935,0.1146
1,shuffle_16,-0.6270,0.2206,-0.2569,0.4258


## 13. Simpan protocol audit

Manifest mencatat checkpoint, seed, architecture, perturbation protocol,
held-out datasets, dan definisi delta.

In [17]:
config = {
    "task": "heldout_temporal_order_evaluation_only",
    "checkpoint_seed": RUN_SEED,
    "variant": VARIANT,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "sequence_length": SEQUENCE_LENGTH,
    "lstm_units": LSTM_UNITS,
    "bidirectional": BIDIRECTIONAL,
    "temporal_context": (
        "concatenate final forward and backward hidden states"
    ),
    "shuffle_chunk_sizes": SHUFFLE_CHUNK_SIZES,
    "shuffle_levels": {
        "shuffle_4": "25% chunks of T=16",
        "shuffle_8": "50% chunks of T=16",
        "shuffle_16": "100% chunk of T=16",
    },
    "num_random_repeats": NUM_RANDOM_REPEATS,
    "permutation_base_seed": PERMUTATION_BASE_SEED,
    "run_reverse": RUN_REVERSE,
    "permutation_rule": (
        "independent deterministic per-video chunk permutation; "
        "SeedSequence(base_seed, repeat, crc32(dataset), "
        "source_idx, chunk_size)"
    ),
    "checkpoint_seed_excluded_from_permutation_generation": True,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "no_cross_dataset_pooling": True,
    "no_training": True,
    "no_fine_tuning": True,
    "no_augmentation": True,
    "use_amp": USE_AMP,
    "primary_metric": "average_precision",
    "delta_definition": (
        "Original minus Perturbed; positive value indicates "
        "performance decrease after temporal-order perturbation"
    ),
    "device": str(
        DEVICE
    ),
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
}

manifest_path = (
    TEST_OUTPUT_DIR
    / f"bilstm_final_temporal_order_protocol_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config,
        file,
        indent=2,
    )

print(
    "Saved all outputs to:",
    TEST_OUTPUT_DIR,
)

print(
    "Protocol:",
    manifest_path,
)

Saved all outputs to: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0
Protocol: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_TemporalOrder/seed_0/bilstm_final_temporal_order_protocol_seed0.json


# Seed berikutnya

Setelah seed pertama selesai, **cukup ubah**:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

kemudian:

```python
RUN_SEED = 123
```

Jangan ubah:

```python
SEQUENCE_LENGTH = 16
LSTM_UNITS = 32
SHUFFLE_CHUNK_SIZES = [4,8,16]
NUM_RANDOM_REPEATS = 10
PERMUTATION_BASE_SEED = 20260809
EVAL_BATCH_SIZE = 4
USE_AMP = False
```

Output otomatis dipisahkan:

```text
HeldOut_Evaluation/
└── BiLSTMFinal_TemporalOrder/
    ├── seed_0/
    ├── seed_42/
    └── seed_123/
```

File utama:
- `metrics_original_seedX.csv`
- `predictions_original_seedX.csv`
- `metrics_reverse_seedX.csv`
- `predictions_reverse_seedX.csv`
- `bilstm_final_temporal_order_metrics_all_repeats_seedX.csv`
- `bilstm_final_temporal_order_predictions_all_repeats_seedX.csv`
- `bilstm_final_temporal_order_summary_seedX.csv`
- `bilstm_final_temporal_order_delta_all_repeats_seedX.csv`
- `bilstm_final_temporal_order_delta_summary_seedX.csv`
- `temporal_order_permutation_audit_examples.json`
- `heldout_dataset_audit.csv`
- `bilstm_final_temporal_order_protocol_seedX.json`